# Answers - StarDist

### First run the main notebook

In [ ]:
%%capture
%run ./stardist.ipynb

## Exercise: What happens if you do not normalize?

Predicting on the raw image finds far more "nuclei", but they are wrong: the model
latches onto the bright foci and on noise, and it misses the dimmer nuclei.

In [ ]:
# Predict on the raw (unnormalized) image, same scale as before
labels_raw, _ = model.predict_instances(img_nuclei, scale=0.5)
print(f"raw image       : {labels_raw.max()} objects")
print(f"normalized image: {labels.max()} objects")

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
_ = axs[0].imshow(img_nuclei, cmap="gray")
_ = axs[0].imshow(labels_raw, cmap=lbl_cmap, alpha=0.5)
_ = axs[0].set_title(f"no normalization: {labels_raw.max()} objects")
_ = axs[1].imshow(img_nuclei, cmap="gray")
_ = axs[1].imshow(labels, cmap=lbl_cmap, alpha=0.5)
_ = axs[1].set_title(f"normalized: {labels.max()} objects")
for ax in axs:
    _ = ax.axis("off")

The network was trained on images with intensities between roughly 0 and 1, while our
raw image contains values between 3 and 208. The numbers entering the network are then
far outside the range it has seen, and the prediction is unreliable.

Normalization also makes the result independent of exposure time: an image taken twice
as bright gives the same input to the network after normalization.

## Exercise: Which scale works best?

Count the objects for a range of scales and look at where the curve flattens out.

In [ ]:
scales = [0.2, 0.3, 0.4, 0.5, 0.6, 0.75, 1.0]
counts = []

for s in scales:
    lab, _ = model.predict_instances(img_nuclei_norm, scale=s)
    counts.append(lab.max())
    print(f"scale {s:4.2f} -> {lab.max():4d} objects")

_ = plt.plot(scales, counts, "o-")
_ = plt.xlabel("scale")
_ = plt.ylabel("Number of detected objects")

Between 0.2 and 0.5 the count barely changes (36 to 38 objects). From 0.6 upwards it
climbs steeply, because single nuclei are broken into fragments.

In [ ]:
# Look at the two ends of the curve next to the value we chose
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, s in zip(axs, [0.25, 0.5, 0.75]):
    lab, _ = model.predict_instances(img_nuclei_norm, scale=s)
    _ = ax.imshow(img_nuclei, cmap="gray")
    _ = ax.imshow(lab, cmap=lbl_cmap, alpha=0.5)
    _ = ax.set_title(f"scale={s}: {lab.max()} objects")
    _ = ax.axis("off")

We can also estimate the scale from the size of the nuclei: they are about 80 pixels
across and the model was trained on nuclei of about 40 pixels, so the image should be
shrunk by about 40/80 = 0.5.

In [ ]:
# Measure the nuclei we found, to check the "100 pixels across" estimate
props = sk.measure.regionprops(labels)
diameters = [p.equivalent_diameter_area for p in props]
print(f"median nucleus diameter: {np.median(diameters):.0f} pixels")
print(f"suggested scale: {40 / np.median(diameters):.2f}")